# E03_RollerBall — DQN на PyTorch

**Урок:** 1.6 курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)  
**Что реализуется:** Deep Q-Network с целевой сетью, буфером воспроизведения и Double DQN.  
**Ожидаемый результат:** средняя награда **> 0.9**, успех 100 %.  
**Время обучения:** ~8 минут на сид (`QUICK_RUN = True` — около 45 секунд).

## Чем этот пример принципиально важнее предыдущего

В `E01_GridWorld` политикой была таблица, и её экспорт в ONNX математически
точен. Здесь политика — **настоящая обучаемая сеть**, и весь путь
«PyTorch → ONNX → Inference Engine в Unity» проверяется по-настоящему.
Именно тут проявляются расхождения порядка наблюдений, нормализации и частоты
принятия решений — то, ради чего в проекте существует требование 10.6.

## Требования

```powershell
python scripts\build_env.py E03_RollerBall
```

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

python        3.10.11
torch         2.2.1+cu121  CUDA 12.1
mlagents_envs 1.2.0.dev0
устройство    cuda
сид           0


## 2. Конфигурация

Все гиперпараметры в одном месте. Разделение намеренное: `DQNConfig` — свойства
**метода**, `DQNTrainConfig` — свойства **процесса обучения** (буфер, частота
обновлений, расписание оценки). Метод не должен знать, как часто его вызывают.

In [2]:
QUICK_RUN = True   # True — быстрая проверка (~45 с); False — полное обучение (~8 мин)

from labrl.algos.dqn import DQNConfig
from labrl.train.dqn import DQNTrainConfig
from labrl.utils.schedules import LinearSchedule

ALGO_CFG = DQNConfig(
    gamma=0.99,                   # горизонт дисконтирования
    learning_rate=5e-4,           # шаг Adam
    batch_size=128,               # переходов в батче
    target_update_interval=500,   # ОБНОВЛЕНИЙ, не шагов среды
    max_grad_norm=10.0,           # клиппинг градиента
    double_dqn=True,              # снимает переоценку Q
)

TOTAL_STEPS = 6_000 if QUICK_RUN else 60_000

TRAIN_CFG = DQNTrainConfig(
    total_steps=TOTAL_STEPS,
    buffer_size=200_000,
    learning_starts=2_000,        # обучение на почти пустом буфере = коррелированная выборка
    train_freq=1,
    gradient_steps=1,
    eval_every_steps=TOTAL_STEPS // 3,
    eval_episodes=20,
)

# ВНИМАНИЕ: расписание разведки НЕ сжимается вместе с бюджетом.
# Сокращённый прогон должен быть началом полного, а не другим экспериментом.
# Измерено: при decay_steps = TOTAL_STEPS*0.5 (то есть 3000) быстрый прогон
# даёт -0.12 вместо 0.94 — с четырьмя тысячами обновлений жадная политика
# ещё плоха, и ранний спад eps лишает буфер разнообразных данных.
# Подробности: docs/07_TROUBLESHOOTING.md, T-10.
EPSILON = LinearSchedule(start=1.0, end=0.05, decay_steps=30_000)

print(f"бюджет: {TOTAL_STEPS} шагов сбора, оценка каждые {TRAIN_CFG.eval_every_steps}")

бюджет: 6000 шагов сбора, оценка каждые 2000


## 3. Подключение среды

In [3]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E03_RollerBall"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

# time_scale выше ~20 ломает физику PhysX: шаг интегрирования становится
# слишком крупным, и среда перестаёт совпадать с тем, что видно в редакторе.
handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")

behavior: E03_RollerBall?team=0
наблюдения (1 сенсоров):
  obs_0: shape=(8,)  имя сенсора: 'VectorSensor_size8'
действия: непрерывных 0, дискретные ветки (4,)

арен (параллельных сред): 8


## 4. Инспекция пространств

`assert` сверяет фактическую среду с `ENV_SPEC.md`. Расхождение обязано падать
здесь, а не проявляться загадочными результатами через час обучения.

In [4]:
OBS_DIM = vec.single_obs_dim
NUM_ACTIONS = int(vec.action_spec.discrete_branches[0])

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim:           {OBS_DIM}")
print(f"дискретные ветки:  {tuple(vec.action_spec.discrete_branches)}")
print(f"непрерывных:       {vec.action_spec.continuous_size}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=8; discrete_branches=4; continuous_size=0; max_step=1000 -->
assert OBS_DIM == 8, f"obs_dim {OBS_DIM} != 8 (ENV_SPEC.md)"
assert NUM_ACTIONS == 4, f"веток {NUM_ACTIONS} != 4 (ENV_SPEC.md)"
assert vec.action_spec.continuous_size == 0
print("\nразмерности совпадают с ENV_SPEC.md")

obs_shapes:        [(8,)]
obs_dim:           8
дискретные ветки:  (4,)
непрерывных:       0

размерности совпадают с ENV_SPEC.md


## 5. Нейросеть

Архитектура определяется **здесь** и полностью редактируема. Алгоритм её
не знает: он принимает любой `nn.Module`, удовлетворяющий протоколу
`QNetwork` — вход `(B, obs_dim)`, выход `(B, num_actions)`.

Замените `MyQNetwork` на что угодно своё — остальной ноутбук не изменится.

In [5]:
from labrl.nets.protocols import check_q_network


class MyQNetwork(nn.Module):
    """Q-сеть: obs (B, 8) -> Q-значения (B, 4)."""

    def __init__(self, obs_dim: int, num_actions: int, hidden: int = 128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, num_actions),
        )

    def forward(self, obs):
        return self.body(obs)


q_net = MyQNetwork(OBS_DIM, NUM_ACTIONS)
target_net = MyQNetwork(OBS_DIM, NUM_ACTIONS)   # та же архитектура, отдельные веса

check_q_network(q_net, OBS_DIM, NUM_ACTIONS)    # проверка формы выхода
print(q_net)
print(f"\nпараметров: {sum(p.numel() for p in q_net.parameters()):,}")

MyQNetwork(
  (body): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

параметров: 18,180


## 6. Теория и математика метода

DQN — это Q-learning, в котором таблица заменена сетью $Q_\theta(s,a)$.
Замена сама по себе не работает: нужны две добавки.

**Буфер воспроизведения** разрывает корреляцию соседних переходов и позволяет
использовать каждый переход многократно — шаг в Unity стоит на порядки дороже
шага оптимизатора.

**Целевая сеть** решает проблему «движущейся цели»: цель обновления зависит
от тех же весов, что и обучаемая величина. Цель считается по замороженной
копии $Q_{\theta^-}$.

Цель (в варианте **Double DQN** — действие выбирает онлайн-сеть, оценивает целевая):

$$
y_t = r_t + \gamma\,(1-\mathrm{terminated}_t)\, Q_{\theta^-}\!\bigl(s_{t+1},\; \arg\max_{a'} Q_\theta(s_{t+1}, a')\bigr)
$$

Функция потерь — Smooth L1 (Huber), а не MSE: большая TD-ошибка в начале
обучения при квадратичной функции даёт градиент, сносящий сеть.

$$
\mathcal{L}(\theta) = \mathrm{SmoothL1}\bigl(Q_\theta(s_t,a_t),\, y_t\bigr)
$$

Ниже — **исходный код обновления**, тот самый, что выполняется при обучении.

In [6]:
import inspect

from labrl.algos.dqn import DQN

print(inspect.getsource(DQN.update))

    def update(self, batch: Batch) -> dict[str, float]:
        """Единственное место, где меняются параметры (требование 8.7).

        Returns:
            Метрики шага обучения для TensorBoard.
        """
        obs = torch.as_tensor(batch.obs, dtype=torch.float32, device=self.device)
        next_obs = torch.as_tensor(batch.next_obs, dtype=torch.float32, device=self.device)
        action = torch.as_tensor(batch.action, dtype=torch.int64, device=self.device)
        reward = torch.as_tensor(batch.reward, dtype=torch.float32, device=self.device)
        terminated = torch.as_tensor(batch.terminated, dtype=torch.bool, device=self.device)

        # Q(s, a) — значение выбранного действия.
        q_values = self.q_net(obs)
        q_selected = q_values.gather(1, action.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            if self.cfg.double_dqn:
                # Double DQN: ДЕЙСТВИЕ выбирает онлайн-сеть, ОЦЕНИВАЕТ — целевая.
                best_action = self.q_net(ne

## 7. Цикл обучения

Цикл импортируется из `labrl.train.dqn`, а не копируется сюда: иначе учебная
и рабочая версии со временем разойдутся. Его исходник печатается целиком.

In [7]:
from labrl.train.dqn import train_dqn

print(inspect.getsource(train_dqn))

def train_dqn(
    vec: VecUnityEnv,
    algo: DQN,
    epsilon_schedule: Schedule,
    logger: TBLogger,
    cfg: DQNTrainConfig | None = None,
    seed: int = 0,
    on_eval: Callable[[int, float, float], None] | None = None,
) -> DQNTrainResult:
    """Обучает DQN в среде Unity.

    Args:
        vec: векторизованная среда (K арен = K параллельных сред).
        algo: алгоритм; его сети меняются на месте.
        epsilon_schedule: расписание ε от номера шага сбора.
        logger: логгер TensorBoard.
        cfg: параметры цикла.
        seed: сид генераторов разведки и выборки из буфера.
        on_eval: колбэк ``(step, mean_reward, success_rate)`` после оценки.
    """
    cfg = cfg or DQNTrainConfig()
    rng = np.random.default_rng(seed)
    result = DQNTrainResult(algo=algo)

    obs = vec.reset()
    n = vec.num_envs
    obs_dim = obs[0].shape[1]
    buffer = ReplayBuffer(cfg.buffer_size, obs_dim, seed=seed)

    running_return = np.zeros(n)
    running_length = np.zeros(n, d

In [8]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger

algo = DQN(q_net, target_net, ALGO_CFG, device=DEVICE)
run = create_run_dir(ENV_ID, "dqn", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_dqn(
    vec=vec, algo=algo, epsilon_schedule=EPSILON, logger=logger,
    cfg=TRAIN_CFG, seed=SEED,
    on_eval=lambda step, r, s: print(f"  шаг {step:>6}: награда {r:+.4f}, успех {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, переходов в буфер: {result.transitions}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

каталог прогона: C:\unity-ml-agents-lab\results\E03_RollerBall\dqn\20260815-200629_seed0



  шаг   2000: награда -0.3802, успех 35%


  шаг   4000: награда +0.7081, успех 90%


  шаг   6000: награда +0.9460, успех 100%

эпизодов: 479, переходов в буфер: 45293
время: 56.1 с
обязательные теги схемы 11.2 без данных: нет


## 8. Оценка и кривая обучения

In [9]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(smoothed)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("награда"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
axes[1].plot(steps, rewards, marker="o")
axes[1].set_title("Eval/Mean Reward (жадная политика)")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"итоговая оценка: награда {result.last_eval_reward:+.4f}, успех {result.last_eval_success:.0%}")
print(f"лучшая за прогон: {result.best_eval_reward:+.4f}")

THRESHOLD = 0.5   # блок success_criteria конфига
assert result.last_eval_reward >= THRESHOLD, (
    f"не достигнут критерий приёмки: {result.last_eval_reward:.4f} < {THRESHOLD}")

итоговая оценка: награда +0.9460, успех 100%
лучшая за прогон: +0.9460


C:\Users\-\AppData\Local\Temp\ipykernel_12072\975980783.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Экспорт в ONNX и верификация

`policy_module()` возвращает саму Q-сеть; обёртка экспорта берёт от неё
`argmax` — ровно ту политику, которую оценивали выше.

Наблюдения для проверки числового паритета собираются **из реальной среды**:
требование 10.5 говорит именно о реальном распределении, и на случайном шуме
расхождение нормализации может не проявиться.

In [10]:
from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из среды под текущей политикой.
collected, obs_now = [], vec.reset()
while sum(len(c) for c in collected) < 64:
    collected.append(obs_now[0].copy())
    obs_now = vec.step(algo.greedy_action(obs_now[0])[:, None].astype(np.int32)).obs
sample = [np.concatenate(collected, axis=0)[:64]]

spec = ActionSpecLite(discrete_branches=(NUM_ACTIONS,))
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

model_dest = REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID / "Models" / f"{ENV_ID}_dqn.onnx"
model_dest.parent.mkdir(parents=True, exist_ok=True)
model_dest.write_bytes(onnx_path.read_bytes())
print(f"\nмодель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")

vec.close()

OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    тип выхода deterministic_discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа discrete_action_output_shape — ожидалось [4.0], получено [4.0]
OK    числовой паритет: deterministic_discrete_actions — max|ONNX-PyTorch| = 0.000e+00, допуск 1e-04
OK    диапазон дискретной ветки 0 — значения [0, 3], допустимо целые [0, 4)
ИТОГ: ПРОЙДЕНО (13/13)

модель скопирована в проект Unity: unity\MLAgentsLab\Assets\Envs\E03_RollerBall\Models\E03_RollerBall_dqn.onnx


## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E03_RollerBall__dqn.yaml --python-reward 0.94
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Порог — 0.8.

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E03_RollerBall/Scenes/E03_RollerBall.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента: `Model` = импортированная модель, `Behavior Type` = `Inference Only`,
   галочка `Deterministic Inference` — Python оценивал именно детерминированную
   политику, и сравнивать надо одно с одним.
5. Play: шар должен целенаправленно катиться к зелёному кубу и не падать с краёв.

### Если награда в Unity заметно ниже

Типовые причины (10.6): рассогласование порядка наблюдений, отсутствующая
нормализация, различие в частоте принятия решений. Разбор конкретных случаев —
`docs/07_TROUBLESHOOTING.md`.

### Просмотр метрик

```powershell
.\scripts\tb.ps1 -EnvId E03_RollerBall
```